In [1]:
# Imports 
from pyspark.sql import SparkSession

In [2]:
# Spark session instantiation 
spark = SparkSession.builder\
        .appName('Joining RDDs')\
        .master('spark://172.22.0.2:7077')\
        .getOrCreate()

sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/05 12:16:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 1. Parts and partSub :

In [3]:
part_input = sc.textFile('/opt/spark-data/tpch-data/part.tbl.gz')
part_input.take(10)

['1|goldenrod lavender spring chocolate lace|Manufacturer#1|Brand#13|PROMO BURNISHED COPPER|7|JUMBO PKG|901.00|ly. slyly ironi|',
 '2|blush thistle blue yellow saddle|Manufacturer#1|Brand#13|LARGE BRUSHED BRASS|1|LG CASE|902.00|lar accounts amo|',
 '3|spring green yellow purple cornsilk|Manufacturer#4|Brand#42|STANDARD POLISHED BRASS|21|WRAP CASE|903.00|egular deposits hag|',
 '4|cornflower chocolate smoke green pink|Manufacturer#3|Brand#34|SMALL PLATED BRASS|14|MED DRUM|904.00|p furiously r|',
 '5|forest brown coral puff cream|Manufacturer#3|Brand#32|STANDARD POLISHED TIN|15|SM PKG|905.00| wake carefully |',
 '6|bisque cornflower lawn forest magenta|Manufacturer#2|Brand#24|PROMO PLATED STEEL|4|MED BAG|906.00|sual a|',
 '7|moccasin green thistle khaki floral|Manufacturer#1|Brand#11|SMALL PLATED COPPER|45|SM BAG|907.00|lyly. ex|',
 '8|misty lace thistle snow royal|Manufacturer#4|Brand#44|PROMO BURNISHED TIN|41|LG DRUM|908.00|eposi|',
 '9|thistle dim navajo dark gainsboro|Manufacturer#4|

In [4]:
part_input_splitted = part_input.map(lambda s : s.split('|'))
part_input_splitted.take(5)

[['1',
  'goldenrod lavender spring chocolate lace',
  'Manufacturer#1',
  'Brand#13',
  'PROMO BURNISHED COPPER',
  '7',
  'JUMBO PKG',
  '901.00',
  'ly. slyly ironi',
  ''],
 ['2',
  'blush thistle blue yellow saddle',
  'Manufacturer#1',
  'Brand#13',
  'LARGE BRUSHED BRASS',
  '1',
  'LG CASE',
  '902.00',
  'lar accounts amo',
  ''],
 ['3',
  'spring green yellow purple cornsilk',
  'Manufacturer#4',
  'Brand#42',
  'STANDARD POLISHED BRASS',
  '21',
  'WRAP CASE',
  '903.00',
  'egular deposits hag',
  ''],
 ['4',
  'cornflower chocolate smoke green pink',
  'Manufacturer#3',
  'Brand#34',
  'SMALL PLATED BRASS',
  '14',
  'MED DRUM',
  '904.00',
  'p furiously r',
  ''],
 ['5',
  'forest brown coral puff cream',
  'Manufacturer#3',
  'Brand#32',
  'STANDARD POLISHED TIN',
  '15',
  'SM PKG',
  '905.00',
  ' wake carefully ',
  '']]

In [5]:
#part_input_splitted_clean = part_input_splitted.map(lambda arr : arr[:-1])
#part_input_splitted_clean.take(5)
PART_SIZE = 10 
part_mapped = part_input_splitted\
    .filter(lambda arr : len(arr) == PART_SIZE) 
print(f'number of accepted parts is : {part_mapped.count()}')

number of accepted parts is : 40000


In [6]:
#catch the rejected records 
part_rejected = part_input_splitted\
    .filter(lambda arr : len(arr) != PART_SIZE) 
print(f'number of rejected parts is : {part_rejected.count()}')


number of rejected parts is : 0


In [7]:
class Part : 
    def __init__(self, partkey, name, mfgr, brand, type_, size, container, retailprice, comment) : 
        self.partkey = partkey          # int
        self.name = name                # str
        self.mfgr = mfgr                # str
        self.brand = brand              # str
        self.type = type_               # str
        self.size = size                # int
        self.container = container      # str
        self.retailprice = retailprice  # float
        self.comment = comment          # str

    def __repr__(self): 
        return (f"Part : {self.partkey}, {self.name}, {self.mfgr}, {self.brand}, {self.type}, {self.size}, {self.container}, {self.retailprice}, {self.comment})")

In [8]:
# columns type casting 

part_transformed = part_mapped.map(lambda arr : ( int(arr[0]), Part(int(arr[0]), arr[1], arr[2], arr[3], arr[4], int(arr[5]), arr[6], float(arr[7]), arr[8])))

In [9]:
part_transformed.take(3)

[(1,
  Part : 1, goldenrod lavender spring chocolate lace, Manufacturer#1, Brand#13, PROMO BURNISHED COPPER, 7, JUMBO PKG, 901.0, ly. slyly ironi)),
 (2,
  Part : 2, blush thistle blue yellow saddle, Manufacturer#1, Brand#13, LARGE BRUSHED BRASS, 1, LG CASE, 902.0, lar accounts amo)),
 (3,
  Part : 3, spring green yellow purple cornsilk, Manufacturer#4, Brand#42, STANDARD POLISHED BRASS, 21, WRAP CASE, 903.0, egular deposits hag))]

### 2. Read and Parse the partsupp dataset : 

In [10]:
class PartSupp : 
    def __init__(self, ps_partkey, ps_suppkey, ps_availqty, ps_supplycost, ps_comment) : 
        self.ps_partkey = ps_partkey        # int
        self.ps_suppkey = ps_suppkey        # int
        self.ps_availqty = ps_availqty      # int
        self.ps_supplycost = ps_supplycost  # float
        self.ps_comment = ps_comment        # str 

    def __repr__(self):
        return (
            f"PartSupp("
            f"{self.ps_partkey}, "
            f"{self.ps_suppkey}, "
            f"{self.ps_availqty}, "
            f"{self.ps_supplycost}, "
            f"{self.ps_comment})"
        )

In [11]:
# read the partsupp input file 

partSupp_input = sc.textFile('/opt/spark-data/tpch-data/partsupp.tbl.gz')
partSupp_input.take(10)

['1|2|3325|771.64|, even theodolites. regular, final theodolites eat after the carefully pending foxes. furiously regular deposits sleep slyly. carefully bold realms above the ironic dependencies haggle careful|',
 '1|502|8076|993.49|ven ideas. quickly even packages print. pending multipliers must have to are fluff|',
 '1|1002|3956|337.09|after the fluffily ironic deposits? blithely special dependencies integrate furiously even excuses. blithely silent theodolites could have to haggle pending, express requests; fu|',
 '1|1502|4069|357.84|al, regular dependencies serve carefully after the quickly final pinto beans. furiously even deposits sleep quickly final, silent pinto beans. fluffily reg|',
 '2|3|8895|378.49|nic accounts. final accounts sleep furiously about the ironic, bold packages. regular, regular accounts|',
 '2|503|4969|915.27|ptotes. quickly pending dependencies integrate furiously. fluffily ironic ideas impress blithely above the express accounts. furiously even epitaphs nee

In [12]:
partsupp_splitted = partSupp_input.map(lambda arr : arr.split('|')[:-1])
partsupp_splitted.take(4)

[['1',
  '2',
  '3325',
  '771.64',
  ', even theodolites. regular, final theodolites eat after the carefully pending foxes. furiously regular deposits sleep slyly. carefully bold realms above the ironic dependencies haggle careful'],
 ['1',
  '502',
  '8076',
  '993.49',
  'ven ideas. quickly even packages print. pending multipliers must have to are fluff'],
 ['1',
  '1002',
  '3956',
  '337.09',
  'after the fluffily ironic deposits? blithely special dependencies integrate furiously even excuses. blithely silent theodolites could have to haggle pending, express requests; fu'],
 ['1',
  '1502',
  '4069',
  '357.84',
  'al, regular dependencies serve carefully after the quickly final pinto beans. furiously even deposits sleep quickly final, silent pinto beans. fluffily reg']]

In [13]:
partsupp_transformed = partsupp_splitted.map(lambda arr : (int(arr[0]), PartSupp(int(arr[0]), int(arr[1]), int(arr[2]), float(arr[3]), str(arr[4])) ) )
partsupp_transformed.take(4)

[(1,
  PartSupp(1, 2, 3325, 771.64, , even theodolites. regular, final theodolites eat after the carefully pending foxes. furiously regular deposits sleep slyly. carefully bold realms above the ironic dependencies haggle careful)),
 (1,
  PartSupp(1, 502, 8076, 993.49, ven ideas. quickly even packages print. pending multipliers must have to are fluff)),
 (1,
  PartSupp(1, 1002, 3956, 337.09, after the fluffily ironic deposits? blithely special dependencies integrate furiously even excuses. blithely silent theodolites could have to haggle pending, express requests; fu)),
 (1,
  PartSupp(1, 1502, 4069, 357.84, al, regular dependencies serve carefully after the quickly final pinto beans. furiously even deposits sleep quickly final, silent pinto beans. fluffily reg))]

### 3. Join part and partSupp : 

In [ ]:
# performn inner join between part_transformed and partsupp_transformed rdds :

# part => rdd [[int, Part]]
# partsupp => rdd ({int, Part})
# part_joining_partSupp  => rdd ({int, Part, PartSupp })
part_joined_partsupp = part_transformed.join(partsupp_transformed)

#take the first 10 records 

for record in part_joined_partsupp.take(1) : 
    print(record)

In [ ]:
part_transformed.count()


In [ ]:
partsupp_transformed.count()

In [ ]:
part_joined_partsupp.count()

### types of joins : 

In [30]:
employees =sc.parallelize( [(1,'ali'),(2,'mohammed'),(2,'hassan'),(3,'omar'),(4,'youssef')])
departments =sc.parallelize( [(1,'Informatique'),(2,'biologie'),(3,'Math'),(0,'unknown')] )

### inner join : 

In [31]:
emp_dep = employees.rightOuterJoin(departments).collect()
print(emp_dep)

[(0, (None, 'unknown')), (1, ('ali', 'Informatique')), (2, ('mohammed', 'biologie')), (2, ('hassan', 'biologie')), (3, ('omar', 'Math'))]
